In [0]:
%sql
-- DESCRIBE DETAIL bfsi_lakehouse.bronze.t_loaninstallment;
DESCRIBE DETAIL bfsi_lakehouse.bronze.t_Client;
-- look at two columns:
--   numFiles  → how many active files RIGHT NOW
--   location  → the actual folder path in storage

In [0]:
%sql
OPTIMIZE bfsi_lakehouse.bronze.t_loaninstallment

In [0]:
%sql
-- DESCRIBE HISTORY bfsi_lakehouse.bronze.t_loaninstallment
-- VACUUM bfsi_lakehouse.bronze.t_loaninstallment DRY RUN

SELECT * FROM bfsi_lakehouse.bronze.t_loaninstallment VERSION AS OF 1;
-- [DELTA_UNSUPPORTED_TIME_TRAVEL_BEYOND_DELETED_FILE_RETENTION_DURATION] Cannot time travel beyond delta.deletedFileRetentionDuration (168 HOURS) set on the table. SQLSTATE: 0AKDC

In [0]:
from pyspark.sql.functions import broadcast, max, sum

# Column Pruning to reduce the size of the dataframes
loan_df = spark.sql("""SELECT 
                            LoanID,
                            DisbursementAmount 
                        FROM bfsi_lakehouse.silver.t_loan""")

loanInstallment_df = spark.sql("""SELECT 
                                        LoanID,
                                        InstallmentNo,
                                        InstallmentAmount 
                                FROM bfsi_lakehouse.silver.t_loanInstallment""")

result_df = loanInstallment_df.join(
                broadcast(loan_df),
                on = "LoanID",
                how = 'left'
                ).groupBy("LoanID").agg(
                    max("InstallmentNo").alias("No_Of_Installments"),
                    sum("InstallmentAmount").alias("Total_Installment_Amount")
                )

result_df.display(10)

In [0]:
from pyspark.sql.functions import broadcast, max, sum

# Column Pruning to reduce the size of the dataframes
loan_df = spark.sql("""SELECT   -- ClientID not available in loan table, so excluding
                            LoanID,
                            DisbursementAmount 
                        FROM bfsi_lakehouse.silver.t_loan""")

# Column pruning + Aggregation
loanInstallment_df = spark.sql("""SELECT 
                                        LoanID,
                                        InstallmentNo,
                                        InstallmentAmount 
                                FROM bfsi_lakehouse.silver.t_loanInstallment""").groupby("LoanID").agg(
                                    max("InstallmentNo").alias("No_Of_Installments"),
                                    sum("InstallmentAmount").alias("Total_Installment_Amount")
                                )

result_df = loan_df.join(
    loanInstallment_df,
    on = 'LoanID',
    how = 'left'
)

result_df.display(10)

In [0]:
from pyspark.sql import functions as F

loan_df = (
        spark.read.table("bfsi_lakehouse.silver.t_loan")
            .select("LoanID","LastUpdatedAt")
        )

loan_df.createOrReplaceTempView("loan_df")
spark.sql("""
          SELECT 
            LoanID,
            LastUpdatedAt,
            row_number() OVER (PARTITION BY LoanID ORDER BY LastUpdatedAt DESC) rn
          FROM loan_df
          QUALIFY rn = 1
          """).show(10)

In [0]:
from pyspark.sql import functions as F

accountTrx_df = (
                spark.read.table("bfsi_lakehouse.silver.t_accounttrx")
                .select("AccountID","Amount","TrxDateTime","LoanID")
                .filter(F.col("LoanID").isNotNull())
).groupBy("AccountID").agg(
    F.count(F.lit(1)).alias("No_Of_Transactions"),
    F.sum("Amount").alias("Total_Amount")
).filter(F.col("No_Of_Transactions") > 100)

accountTrx_df.show(10)


In [0]:
from pyspark.sql import functions as F

installment_df = (
    spark.read.table("bfsi_lakehouse.silver.t_loaninstallment")
    .select("LoanID","InstallmentDate","PaidStatus")
    .filter(F.col("PaidStatus") != "PENDING")
).groupBy("LoanID").agg(
    F.max("InstallmentDate").alias("LastPaidDate")
)

installment_df.show(10)